In [ ]:
import numpy as np
import pandas as pd
from pymatgen.core import Structure
from matminer.featurizers.base import MultipleFeaturizer
from matminer.featurizers.composition import ElementProperty, Stoichiometry, ValenceOrbital, IonProperty
from matminer.featurizers.structure import (SiteStatsFingerprint, StructuralHeterogeneity,
                                            ChemicalOrdering, StructureComposition, MaximumPackingEfficiency)
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score,roc_curve,auc
import seaborn as sns

# 1. Read the dataset based on HTC

In [7]:
df = pd.read_json('./data/MP_dataset_C12_HTC.json')
df

,nsites,nelements,formula_pretty,formula_anonymous,volume,density,material_id,energy_per_atom,formation_energy_per_atom,energy_above_hull,band_gap,is_metal,structure_json,label,k_ao,k_log
0,14,3,Cs(SbS2)2,AB2C4,416.924169,4.020155,mp-8890,-16.138279,-0.616589,0.000000,1.4413,False,"{'@module': 'pymatgen.core.structure', '@class...",1,0.252145,-0.598350
1,5,3,KGeBr3,ABC3,175.640517,3.322678,mp-998412,-3.495008,-1.412312,0.025231,0.9083,False,"{'@module': 'pymatgen.core.structure', '@class...",1,0.186959,-0.728253
2,4,3,LiAgF2,ABC2,51.295990,4.946592,mp-1176792,-9.631004,-2.133530,0.023437,1.8912,False,"{'@module': 'pymatgen.core.structure', '@class...",1,0.413129,-0.383914
3,6,2,Sb2Os,AB2,130.076043,11.074427,mp-2695,-34.883281,-0.083994,0.000000,0.3447,False,"{'@module': 'pymatgen.core.structure', '@class...",0,19.029568,1.279429
4,12,3,LiAgF4,ABC4,150.435277,4.212250,mp-752768,-7.883206,-1.843673,0.035738,1.1186,False,"{'@module': 'pymatgen.core.structure', '@class...",1,0.603420,-0.219381
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
656,14,3,In5AgSe8,AB5C8,408.163709,5.344296,mp-1224092,-3.944965,-0.758815,0.000000,0.2236,False,"{'@module': 'pymatgen.core.structure', '@class...",1,0.102742,-0.988251
657,8,3,GeTe4Pb3,AB3C4,269.026576,7.435517,mp-1224448,-35.894271,-0.344737,0.042003,0.6227,False,"{'@module': 'pymatgen.core.structure', '@class...",0,3.540820,0.549104
658,7,3,Tl3AsS3,AB3C3,193.649007,6.725080,mp-9791,-3.978961,-0.521873,0.006070,1.2156,False,"{'@module': 'pymatgen.core.structure', '@class...",0,0.107991,-0.966613
659,18,3,Li2PdF6,AB2C6,189.318076,4.110032,mp-13985,-7.451218,-2.287240,0.000000,1.6749,False,"{'@module': 'pymatgen.core.structure', '@class...",1,1.613321,0.207721


In [8]:
df['structure'] = df['structure_json'].apply(lambda x: Structure.from_dict(x, fmt='json'))

In [9]:
df.columns

Index(['nsites', 'nelements', 'formula_pretty', 'formula_anonymous', 'volume',
       'density', 'material_id', 'energy_per_atom',
       'formation_energy_per_atom', 'energy_above_hull', 'band_gap',
       'is_metal', 'structure_json', 'label', 'k_ao', 'k_log', 'structure'],
      dtype='object')

# 2. Feature generation

In [10]:
featurizer = MultipleFeaturizer([
    SiteStatsFingerprint.from_preset("CoordinationNumber_ward-prb-2017"),
    StructuralHeterogeneity(),
    ChemicalOrdering(),  
    
    MaximumPackingEfficiency(),
    SiteStatsFingerprint.from_preset("LocalPropertyDifference_ward-prb-2017"),
    StructureComposition(Stoichiometry()),
    StructureComposition(ElementProperty.from_preset("magpie")),
    StructureComposition(ValenceOrbital(props=['frac'])),
    StructureComposition(IonProperty(fast=True))
])

df_fea = featurizer.featurize_dataframe(df, col_id="structure")
df_fea.head()

MultipleFeaturizer:   0%|          | 0/661 [00:00<?, ?it/s]

,nsites,nelements,formula_pretty,formula_anonymous,volume,density,material_id,energy_per_atom,formation_energy_per_atom,energy_above_hull,...,MagpieData mean SpaceGroupNumber,MagpieData avg_dev SpaceGroupNumber,MagpieData mode SpaceGroupNumber,frac s valence electrons,frac p valence electrons,frac d valence electrons,frac f valence electrons,compound possible,max ionic char,avg ionic char
0,14,3,Cs(SbS2)2,AB2C4,416.924169,4.020155,mp-8890,-16.138279,-0.616589,0.000000,...,120.142857,57.306122,70.0,0.236364,0.400000,0.363636,0.000000,False,0.551131,0.069434
1,5,3,KGeBr3,ABC3,175.640517,3.322678,mp-998412,-3.495008,-1.412312,0.025231,...,129.200000,78.240000,64.0,0.136364,0.257576,0.606061,0.000000,True,0.681744,0.117973
2,4,3,LiAgF2,ABC2,51.295990,4.946592,mp-1176792,-9.631004,-2.133530,0.023437,...,121.000000,106.000000,15.0,0.230769,0.384615,0.384615,0.000000,True,0.894601,0.205734
3,6,2,Sb2Os,AB2,130.076043,11.074427,mp-2695,-34.883281,-0.083994,0.000000,...,175.333333,12.444444,166.0,0.115385,0.115385,0.500000,0.269231,False,0.005609,0.001246
4,12,3,LiAgF4,ABC4,150.435277,4.212250,mp-752768,-7.883206,-1.843673,0.035738,...,85.666667,94.222222,15.0,0.250000,0.500000,0.250000,0.000000,False,0.894601,0.177264


In [11]:
df_fea['k_class'] = 1

df_fea.loc[df_fea['k_ao'] > 2, 'k_class'] = 0

In [12]:
count_0 = (df_fea['k_class'] == 0).sum()
count_0

217

In [13]:
count_1 = (df_fea['k_class'] == 1).sum()
count_1

444

In [14]:
X = df_fea.drop(['nsites', 'nelements', 'formula_pretty', 'formula_anonymous', 'volume',
       'density', 'material_id', 'energy_per_atom',
       'formation_energy_per_atom', 'energy_above_hull', 'band_gap',
       'is_metal', 'structure_json', 'label', 'k_ao', 'k_log', 'structure',
                'k_class'], axis = 1)

X

,minimum CN_VoronoiNN,maximum CN_VoronoiNN,range CN_VoronoiNN,mean CN_VoronoiNN,avg_dev CN_VoronoiNN,mean absolute deviation in relative bond length,max relative bond length,min relative bond length,minimum neighbor distance variation,maximum neighbor distance variation,...,MagpieData mean SpaceGroupNumber,MagpieData avg_dev SpaceGroupNumber,MagpieData mode SpaceGroupNumber,frac s valence electrons,frac p valence electrons,frac d valence electrons,frac f valence electrons,compound possible,max ionic char,avg ionic char
0,7.451238,11.852844,4.401606,9.317722,1.006927,0.042590,1.130923,0.924097,5.462055e-02,0.202398,...,120.142857,57.306122,70.0,0.236364,0.400000,0.363636,0.000000,False,0.551131,0.069434
1,5.969593,12.005647,6.036054,10.608924,1.855732,0.084987,1.117352,0.787531,1.209678e-02,0.123635,...,129.200000,78.240000,64.0,0.136364,0.257576,0.606061,0.000000,True,0.681744,0.117973
2,6.000000,7.420072,1.420072,6.935351,0.484721,0.064891,1.115926,0.870218,1.054413e-07,0.178886,...,121.000000,106.000000,15.0,0.230769,0.384615,0.384615,0.000000,True,0.894601,0.205734
3,7.211076,9.414104,2.203028,8.679761,0.979123,0.053056,1.039792,0.920416,4.406832e-02,0.127974,...,175.333333,12.444444,166.0,0.115385,0.115385,0.500000,0.269231,False,0.005609,0.001246
4,7.100904,11.240964,4.140060,10.010793,1.640227,0.054177,1.040632,0.903046,1.442192e-16,0.157342,...,85.666667,94.222222,15.0,0.250000,0.500000,0.250000,0.000000,False,0.894601,0.177264
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
656,5.741111,6.818940,1.077829,6.311252,0.384054,0.040734,1.048487,0.944346,1.559733e-01,0.203981,...,73.714286,68.244898,14.0,0.132353,0.181373,0.686275,0.000000,True,0.137763,0.031998
657,5.996431,6.443000,0.446569,6.273414,0.137653,0.007729,1.021094,0.973891,5.657788e-03,0.039081,...,188.500000,36.500000,152.0,0.098765,0.148148,0.493827,0.259259,True,0.025275,0.003775
658,7.077826,11.605190,4.527365,10.017743,1.360668,0.056295,1.065678,0.881517,9.032000e-02,0.258359,...,136.857143,57.306122,70.0,0.122807,0.157895,0.350877,0.368421,True,0.205784,0.044814
659,5.999997,11.824808,5.824811,9.781023,2.515332,0.099357,1.078966,0.830062,6.322125e-04,0.170943,...,85.888889,94.518519,15.0,0.259259,0.555556,0.185185,0.000000,True,0.894601,0.180732


In [15]:
y = df_fea["k_class"]
y

0      1
1      1
2      1
3      0
4      1
      ..
656    1
657    0
658    1
659    1
660    1
Name: k_class, Length: 661, dtype: int64

# 3. Compare different models based on 10-fold cross validation

In [17]:
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import StratifiedKFold

kfold = StratifiedKFold(n_splits=10, random_state=19981024, shuffle=True)

In [18]:
from sklearn.ensemble import RandomForestClassifier

clf = RandomForestClassifier(random_state=1998)
scores = cross_val_score(clf, X, y, cv=kfold, scoring = 'roc_auc')
accu = cross_val_score(clf, X, y, cv=kfold, scoring = 'accuracy')
#print(scores)
print(scores.mean())
print(accu.mean())

0.8878433687524596
0.8245816372682044


In [19]:
from xgboost import XGBClassifier

clf = XGBClassifier(random_state=1998)
scores = cross_val_score(clf, X, y, cv=kfold, scoring = 'roc_auc')
accu = cross_val_score(clf, X, y, cv=kfold, scoring = 'accuracy')
#print(scores)
print(scores.mean())
print(accu.mean())

0.873856967947877
0.8154681139755766


In [34]:
from sklearn.ensemble import GradientBoostingClassifier
## 定义 GBC模型 
clf =GradientBoostingClassifier(random_state=1998)
scores = cross_val_score(clf, X, y, cv=kfold, scoring = 'roc_auc')
accu = cross_val_score(clf, X, y, cv=kfold, scoring = 'accuracy')
#print(scores)
print(scores.mean())
print(accu.mean())

0.883107044470681
0.8215287200361827


In [35]:
from lightgbm import LGBMClassifier

## 定义 LightGBM 模型 
clf = LGBMClassifier(random_state=1998)
#scores = cross_val_score(clf, X_ss, y, cv=kfold, scoring = 'accuracy')
scores = cross_val_score(clf, X, y, cv=kfold, scoring = 'roc_auc')
accu = cross_val_score(clf, X, y, cv=kfold, scoring = 'accuracy')
#print(scores)
print(scores.mean())
print(accu.mean())

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 195, number of negative: 399
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.011336 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 27336
[LightGBM] [Info] Number of data points in the train set: 594, number of used features: 244
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.328283 -> initscore=-0.715962
[LightGBM] [Info] Start training from score -0.715962
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive ga

In [37]:
from sklearn.tree import DecisionTreeClassifier

clf = DecisionTreeClassifier(random_state=1998)
scores = cross_val_score(clf, X, y, cv=kfold, scoring = 'roc_auc')
accu = cross_val_score(clf, X, y, cv=kfold, scoring = 'accuracy')
#print(scores)
print(scores.mean())
print(accu.mean())

0.7496608946608947
0.7805970149253731


In [38]:
from sklearn.ensemble import ExtraTreesClassifier

clf = ExtraTreesClassifier(random_state=1998)
scores = cross_val_score(clf, X, y, cv=kfold, scoring = 'roc_auc')
accu = cross_val_score(clf, X, y, cv=kfold, scoring = 'accuracy')
#print(scores)
print(scores.mean())
print(accu.mean())

0.8761862674362675
0.815513342379014


In [16]:
from sklearn.ensemble import AdaBoostClassifier

clf = AdaBoostClassifier(random_state=1998)
scores = cross_val_score(clf, X, y, cv=kfold, scoring = 'roc_auc')
accu = cross_val_score(clf, X, y, cv=kfold, scoring = 'accuracy')
#print(scores)
print(scores.mean())
print(accu.mean())

0.8663123660850933
0.8184305744007236


In [15]:
from catboost import CatBoostClassifier

clf = CatBoostClassifier(random_state=1998)
scores = cross_val_score(clf, X, y, cv=kfold, scoring = 'roc_auc')
accu = cross_val_score(clf, X, y, cv=kfold, scoring = 'accuracy')
#print(scores)
print(scores.mean())
print(accu.mean())

Learning rate set to 0.008248
0:	learn: 0.6889671	total: 83ms	remaining: 1m 22s
1:	learn: 0.6845493	total: 90.8ms	remaining: 45.3s
2:	learn: 0.6792496	total: 97.8ms	remaining: 32.5s
3:	learn: 0.6741088	total: 104ms	remaining: 26s
4:	learn: 0.6695852	total: 111ms	remaining: 22.1s
5:	learn: 0.6653066	total: 118ms	remaining: 19.5s
6:	learn: 0.6602068	total: 124ms	remaining: 17.6s
7:	learn: 0.6559006	total: 130ms	remaining: 16.1s
8:	learn: 0.6524844	total: 136ms	remaining: 15s
9:	learn: 0.6488114	total: 141ms	remaining: 14s
10:	learn: 0.6434232	total: 148ms	remaining: 13.3s
11:	learn: 0.6394151	total: 154ms	remaining: 12.7s
12:	learn: 0.6352490	total: 160ms	remaining: 12.2s
13:	learn: 0.6305561	total: 166ms	remaining: 11.7s
14:	learn: 0.6264968	total: 172ms	remaining: 11.3s
15:	learn: 0.6227788	total: 178ms	remaining: 10.9s
16:	learn: 0.6182830	total: 184ms	remaining: 10.6s
17:	learn: 0.6144694	total: 190ms	remaining: 10.4s
18:	learn: 0.6102869	total: 196ms	remaining: 10.1s
19:	learn: 0.60

In [20]:
from sklearn.svm import SVC

ss = StandardScaler()
X_ss = ss.fit_transform(X)
clf = SVC(random_state=1998, kernel='rbf')
scores = cross_val_score(clf, X_ss, y, cv=kfold, scoring = 'roc_auc')
accu = cross_val_score(clf, X_ss, y, cv=kfold, scoring = 'accuracy')
#print(scores)
print(scores.mean())
print(accu.mean())

0.8788306222397132
0.8063998190863864


# 4. Feature selection

## 4.1 Model-based wrapper method  
Descriptors are iteratively removed based on the feature importance scores obtained from the CatBoost algorithm, and an ML model is subsequently trained using the remaining descriptors at each step

In [23]:
cab = CatBoostClassifier(random_state=1998) 
model = cab.fit(X, y)

Learning rate set to 0.008633
0:	learn: 0.6885815	total: 96.5ms	remaining: 1m 36s
1:	learn: 0.6842017	total: 105ms	remaining: 52.2s
2:	learn: 0.6795803	total: 113ms	remaining: 37.4s
3:	learn: 0.6736846	total: 120ms	remaining: 29.9s
4:	learn: 0.6694115	total: 127ms	remaining: 25.4s
5:	learn: 0.6654192	total: 135ms	remaining: 22.4s
6:	learn: 0.6606206	total: 143ms	remaining: 20.2s
7:	learn: 0.6555866	total: 150ms	remaining: 18.7s
8:	learn: 0.6515120	total: 158ms	remaining: 17.4s
9:	learn: 0.6478855	total: 166ms	remaining: 16.4s
10:	learn: 0.6428744	total: 173ms	remaining: 15.5s
11:	learn: 0.6393260	total: 180ms	remaining: 14.8s
12:	learn: 0.6358441	total: 186ms	remaining: 14.2s
13:	learn: 0.6309978	total: 194ms	remaining: 13.7s
14:	learn: 0.6269680	total: 201ms	remaining: 13.2s
15:	learn: 0.6231725	total: 207ms	remaining: 12.7s
16:	learn: 0.6192011	total: 214ms	remaining: 12.4s
17:	learn: 0.6154941	total: 221ms	remaining: 12s
18:	learn: 0.6121480	total: 228ms	remaining: 11.7s
19:	learn: 

### Save the feature importance

In [26]:
Bset = pd.DataFrame()
Bset['feature'] = X.columns
Bset['importance'] = model.feature_importances_
Bset = Bset.sort_values(by='importance', ascending=False)
Bset.reset_index(drop=True, inplace=True)
Bset

,feature,importance
0,minimum neighbor distance variation,2.396339
1,MagpieData minimum AtomicWeight,1.853504
2,max packing efficiency,1.716766
3,MagpieData maximum CovalentRadius,1.695726
4,MagpieData mean Number,1.393513
...,...,...
268,MagpieData minimum NpValence,0.000000
269,MagpieData minimum GSbandgap,0.000000
270,MagpieData minimum NpUnfilled,0.000000
271,minimum local difference in NfUnfilled,0.000000


Considering that writing a for loop and running it once may cause CatBoost memory overflow, it is recommended to run it step by step

In [28]:
selected_features = Bset['feature'].head(11).tolist()
X_selected = X[selected_features]

clf = CatBoostClassifier(random_state=1998)
scores = cross_val_score(clf, X_selected, y, cv=kfold, scoring = 'roc_auc')
roc_mean = np.mean(scores)

print(f'ten-fold scores: ROC_mean={roc_mean:.2f}')

Learning rate set to 0.008248
0:	learn: 0.6876764	total: 6.16ms	remaining: 6.15s
1:	learn: 0.6827019	total: 9.81ms	remaining: 4.9s
2:	learn: 0.6768219	total: 13ms	remaining: 4.32s
3:	learn: 0.6718565	total: 16ms	remaining: 4s
4:	learn: 0.6664216	total: 18.9ms	remaining: 3.77s
5:	learn: 0.6612231	total: 21.6ms	remaining: 3.58s
6:	learn: 0.6569157	total: 24.3ms	remaining: 3.44s
7:	learn: 0.6527366	total: 27ms	remaining: 3.34s
8:	learn: 0.6483948	total: 29.7ms	remaining: 3.27s
9:	learn: 0.6435155	total: 32.4ms	remaining: 3.21s
10:	learn: 0.6393926	total: 35.1ms	remaining: 3.15s
11:	learn: 0.6347297	total: 37.7ms	remaining: 3.1s
12:	learn: 0.6306701	total: 40.2ms	remaining: 3.05s
13:	learn: 0.6262482	total: 42.8ms	remaining: 3.01s
14:	learn: 0.6213852	total: 45.3ms	remaining: 2.98s
15:	learn: 0.6179424	total: 47.8ms	remaining: 2.94s
16:	learn: 0.6136588	total: 50ms	remaining: 2.89s
17:	learn: 0.6097412	total: 52.4ms	remaining: 2.86s
18:	learn: 0.6056145	total: 54.6ms	remaining: 2.82s
19:	l

## 4.2 Pearson correlation
For pairs of features with $\left| p \right|$ > 0.8, we retain the one with the higher feature importance score

In [29]:
selected_features = Bset['feature'].head(11).tolist()
X_s = X[selected_features]
X_s

,minimum neighbor distance variation,MagpieData minimum AtomicWeight,max packing efficiency,MagpieData maximum CovalentRadius,MagpieData mean Number,MagpieData minimum Number,min relative bond length,mean neighbor distance variation,MagpieData mean Row,MagpieData avg_dev MeltingT,MagpieData avg_dev Electronegativity
0,5.462055e-02,32.0650,0.311138,244.0,31.571429,16.0,0.924097,0.159347,4.000000,217.458776,0.465306
1,1.209678e-02,39.0983,0.384463,203.0,31.200000,19.0,0.787531,0.090941,4.000000,296.933600,0.741600
2,1.054413e-07,6.9410,0.363430,145.0,17.000000,3.0,0.870218,0.132359,2.750000,395.405000,1.262500
3,4.406832e-02,121.7600,0.446026,144.0,59.333333,51.0,0.920416,0.100005,5.333333,1067.653333,0.066667
4,1.442192e-16,6.9410,0.353018,145.0,14.333333,3.0,0.903046,0.102068,2.500000,351.471111,1.122222
...,...,...,...,...,...,...,...,...,...,...,...
656,1.559733e-01,78.9600,0.312321,145.0,40.285714,34.0,0.944346,0.183286,4.428571,101.564694,0.364898
657,5.657788e-03,72.6400,0.488542,146.0,60.750000,32.0,0.973891,0.025433,5.250000,118.354063,0.116250
658,9.032000e-02,32.0650,0.354918,145.0,46.285714,16.0,0.881517,0.136979,4.428571,155.211429,0.421224
659,6.322125e-04,6.9410,0.360448,139.0,11.777778,3.0,0.830062,0.115505,2.333333,381.471111,1.152593


In [30]:
cab = CatBoostClassifier(random_state=1998) 
model = cab.fit(X_s, y)

Learning rate set to 0.008633
0:	learn: 0.6868063	total: 4.98ms	remaining: 4.98s
1:	learn: 0.6812928	total: 8.23ms	remaining: 4.11s
2:	learn: 0.6758978	total: 11.3ms	remaining: 3.75s
3:	learn: 0.6705456	total: 14.2ms	remaining: 3.54s
4:	learn: 0.6649557	total: 17.1ms	remaining: 3.4s
5:	learn: 0.6592224	total: 20ms	remaining: 3.31s
6:	learn: 0.6550556	total: 22.7ms	remaining: 3.21s
7:	learn: 0.6509005	total: 25.4ms	remaining: 3.15s
8:	learn: 0.6467368	total: 28.2ms	remaining: 3.1s
9:	learn: 0.6417643	total: 30.7ms	remaining: 3.04s
10:	learn: 0.6365030	total: 33.1ms	remaining: 2.98s
11:	learn: 0.6316053	total: 35.6ms	remaining: 2.93s
12:	learn: 0.6263937	total: 38.2ms	remaining: 2.9s
13:	learn: 0.6217043	total: 40.6ms	remaining: 2.86s
14:	learn: 0.6169154	total: 43ms	remaining: 2.82s
15:	learn: 0.6130683	total: 45.4ms	remaining: 2.79s
16:	learn: 0.6086630	total: 47.8ms	remaining: 2.76s
17:	learn: 0.6048446	total: 50.5ms	remaining: 2.75s
18:	learn: 0.6007322	total: 52.8ms	remaining: 2.73s

In [31]:
Bset = pd.DataFrame()
Bset['feature'] = X_s.columns
Bset['importance'] = model.feature_importances_
Bset = Bset.sort_values(by='importance', ascending=False)
Bset.reset_index(drop=True, inplace=True)
Bset

,feature,importance
0,MagpieData avg_dev MeltingT,15.175345
1,MagpieData avg_dev Electronegativity,12.902845
2,MagpieData maximum CovalentRadius,9.739166
3,min relative bond length,9.207217
4,max packing efficiency,8.664929
5,MagpieData minimum AtomicWeight,8.521726
6,minimum neighbor distance variation,8.020104
7,MagpieData minimum Number,7.232394
8,MagpieData mean Number,7.172352
9,mean neighbor distance variation,7.064391


In [32]:
correlation_matrix = X_s.corr()

highly_correlated_pairs = []
for i in range(len(correlation_matrix.columns)):
    for j in range(i):
        if abs(correlation_matrix.iloc[i, j]) > 0.8:
            highly_correlated_pairs.append((correlation_matrix.columns[i], correlation_matrix.columns[j]))

for pair in highly_correlated_pairs:
    feature1, feature2 = pair
    
    if feature1 in X_s.columns and feature2 in X_s.columns:
        importance1 = Bset.loc[Bset['feature'] == feature1, 'importance'].values[0]
        importance2 = Bset.loc[Bset['feature'] == feature2, 'importance'].values[0]
        
        if importance1 > importance2:
            X_s = X_s.drop(columns=[feature2])
        else:
            X_s = X_s.drop(columns=[feature1])

X_s

,minimum neighbor distance variation,MagpieData minimum AtomicWeight,max packing efficiency,MagpieData maximum CovalentRadius,MagpieData mean Number,min relative bond length,mean neighbor distance variation,MagpieData avg_dev MeltingT,MagpieData avg_dev Electronegativity
0,5.462055e-02,32.0650,0.311138,244.0,31.571429,0.924097,0.159347,217.458776,0.465306
1,1.209678e-02,39.0983,0.384463,203.0,31.200000,0.787531,0.090941,296.933600,0.741600
2,1.054413e-07,6.9410,0.363430,145.0,17.000000,0.870218,0.132359,395.405000,1.262500
3,4.406832e-02,121.7600,0.446026,144.0,59.333333,0.920416,0.100005,1067.653333,0.066667
4,1.442192e-16,6.9410,0.353018,145.0,14.333333,0.903046,0.102068,351.471111,1.122222
...,...,...,...,...,...,...,...,...,...
656,1.559733e-01,78.9600,0.312321,145.0,40.285714,0.944346,0.183286,101.564694,0.364898
657,5.657788e-03,72.6400,0.488542,146.0,60.750000,0.973891,0.025433,118.354063,0.116250
658,9.032000e-02,32.0650,0.354918,145.0,46.285714,0.881517,0.136979,155.211429,0.421224
659,6.322125e-04,6.9410,0.360448,139.0,11.777778,0.830062,0.115505,381.471111,1.152593


In [35]:
df_ = pd.concat([df, X_s], axis=1)
df_['k_class'] = 1
df_.loc[df_['k_ao'] > 2, 'k_class'] = 0
df_

,nsites,nelements,formula_pretty,formula_anonymous,volume,density,material_id,energy_per_atom,formation_energy_per_atom,energy_above_hull,...,minimum neighbor distance variation,MagpieData minimum AtomicWeight,max packing efficiency,MagpieData maximum CovalentRadius,MagpieData mean Number,min relative bond length,mean neighbor distance variation,MagpieData avg_dev MeltingT,MagpieData avg_dev Electronegativity,k_class
0,14,3,Cs(SbS2)2,AB2C4,416.924169,4.020155,mp-8890,-16.138279,-0.616589,0.000000,...,5.462055e-02,32.0650,0.311138,244.0,31.571429,0.924097,0.159347,217.458776,0.465306,1
1,5,3,KGeBr3,ABC3,175.640517,3.322678,mp-998412,-3.495008,-1.412312,0.025231,...,1.209678e-02,39.0983,0.384463,203.0,31.200000,0.787531,0.090941,296.933600,0.741600,1
2,4,3,LiAgF2,ABC2,51.295990,4.946592,mp-1176792,-9.631004,-2.133530,0.023437,...,1.054413e-07,6.9410,0.363430,145.0,17.000000,0.870218,0.132359,395.405000,1.262500,1
3,6,2,Sb2Os,AB2,130.076043,11.074427,mp-2695,-34.883281,-0.083994,0.000000,...,4.406832e-02,121.7600,0.446026,144.0,59.333333,0.920416,0.100005,1067.653333,0.066667,0
4,12,3,LiAgF4,ABC4,150.435277,4.212250,mp-752768,-7.883206,-1.843673,0.035738,...,1.442192e-16,6.9410,0.353018,145.0,14.333333,0.903046,0.102068,351.471111,1.122222,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
656,14,3,In5AgSe8,AB5C8,408.163709,5.344296,mp-1224092,-3.944965,-0.758815,0.000000,...,1.559733e-01,78.9600,0.312321,145.0,40.285714,0.944346,0.183286,101.564694,0.364898,1
657,8,3,GeTe4Pb3,AB3C4,269.026576,7.435517,mp-1224448,-35.894271,-0.344737,0.042003,...,5.657788e-03,72.6400,0.488542,146.0,60.750000,0.973891,0.025433,118.354063,0.116250,0
658,7,3,Tl3AsS3,AB3C3,193.649007,6.725080,mp-9791,-3.978961,-0.521873,0.006070,...,9.032000e-02,32.0650,0.354918,145.0,46.285714,0.881517,0.136979,155.211429,0.421224,1
659,18,3,Li2PdF6,AB2C6,189.318076,4.110032,mp-13985,-7.451218,-2.287240,0.000000,...,6.322125e-04,6.9410,0.360448,139.0,11.777778,0.830062,0.115505,381.471111,1.152593,1


In [37]:
df_ = df_.drop(['structure'], axis=1)

df_.to_json('./data/MP_C12_selected_features.json')